In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt # plotting
import os # accessing directory structure
from mpl_toolkits.mplot3d import Axes3D
from sklearn.preprocessing import StandardScaler

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Download latest version
path = kagglehub.dataset_download("jeanmidev/smart-meters-in-london")

# London Smart Meters - Load Pattern Clustering
**Author:** Gabriele Costantino  
**Date:** July 2026  

## Overview
This notebook applies clustering algorithms to residential electricity consumption 
profiles from the London Smart Meters dataset, with the goal of identifying 
common behavioral patterns among end users.

## Dataset
- Source: Kaggle - London Smart Meters
- Period: 2011-2014 (2013 is the only year that has been considered)
- Users: ~5,500 residential customers
- Granularity: Half-hourly consumption data

## Scope of this work
This project aims to find similar characteristics between different clients and grouping them into coherent clusters, using the k-means clustering algorithm. The clustering results will then be compared to the existing ACORN classification of each client to verify if the different classification successfully translate to a different load consumption.

In [2]:
# Load all blocks
block_files = sorted([f for f in os.listdir('/kaggle/input/datasets/jeanmidev/smart-meters-in-london/halfhourly_dataset/halfhourly_dataset') if f.startswith('block_')])[:100]
print(f"Loading {len(block_files)} blocks...")

df_tot = pd.concat([
    pd.read_csv(os.path.join('/kaggle/input/datasets/jeanmidev/smart-meters-in-london/halfhourly_dataset/halfhourly_dataset', f), dtype=str) 
    for f in block_files
], ignore_index=True)

df_tot.columns = ['LCLid', 'DateTime', 'KWh']
df_tot['DateTime'] = pd.to_datetime(df_tot['DateTime'])

# Find clients with full 12 months in 2013
df_2013_all = df_tot[df_tot['DateTime'].dt.year == 2013]
months_per_client = df_2013_all.groupby('LCLid')['DateTime'].apply(
    lambda x: x.dt.month.nunique()
)
full_year_clients = set(months_per_client[months_per_client == 12].index)

# Find clients present in border periods
clients_border_start = set(df_tot[
    (df_tot['DateTime'].dt.year == 2012) & 
    (df_tot['DateTime'].dt.dayofyear >= 356)
]['LCLid'].unique())

clients_border_end = set(df_tot[
    (df_tot['DateTime'].dt.year == 2014) & 
    (df_tot['DateTime'].dt.dayofyear <= 9)
]['LCLid'].unique())

# Keep only clients present in ALL three periods
valid_clients = full_year_clients & clients_border_start & clients_border_end
print(f"Valid clients (present in all periods): {len(valid_clients)}")

# Filter all three dataframes
df = df_tot[
    (df_tot['DateTime'].dt.year == 2013) & 
    (df_tot['LCLid'].isin(valid_clients))
].copy()

border_start = df_tot[
    (df_tot['DateTime'].dt.year == 2012) & 
    (df_tot['DateTime'].dt.dayofyear >= 356) &
    (df_tot['LCLid'].isin(valid_clients))
].copy()

border_end = df_tot[
    (df_tot['DateTime'].dt.year == 2014) & 
    (df_tot['DateTime'].dt.dayofyear <= 9) &
    (df_tot['LCLid'].isin(valid_clients))
].copy()

del df_tot  # free memory

print(f"Clients in df: {df['LCLid'].nunique()}")
print(f"Clients in border_start: {border_start['LCLid'].nunique()}")
print(f"Clients in border_end: {border_end['LCLid'].nunique()}")
print(f"Total rows df: {len(df)}")

Loading 100 blocks...
Valid clients (present in all periods): 4534
Clients in df: 4534
Clients in border_start: 4534
Clients in border_end: 4534
Total rows df: 79335111


## Data Cleaning
To ensure a successful clustering, a dataset cleaning process is essential.

### 1) Holidays and weekends removal
Working days are the only days included in this analysis, hence the weekends and holidays have been removed from the dataset.

In [3]:
# importing the uk_bank_holidays.csv

bank_holidays = pd.read_csv('/kaggle/input/datasets/jeanmidev/smart-meters-in-london/uk_bank_holidays.csv')
df['DateTime'] = pd.to_datetime(df['DateTime'])

# Weekend filtering (5=Saturday, 6=Sunday)
df_workdays = df[df['DateTime'].dt.dayofweek < 5].copy()

# Bank holidays filtering
bank_holidays['Bank holidays'] = pd.to_datetime(bank_holidays['Bank holidays'])
holiday_dates = bank_holidays['Bank holidays'].dt.date

df_workdays = df_workdays[~df_workdays['DateTime'].dt.date.isin(holiday_dates)].copy()

print(f"Rows before filtering: {len(df)}")
print(f"Rows after filtering: {len(df_workdays)}")
del df  # free up memory

Rows before filtering: 79335111
Rows after filtering: 54991233


### 2) Non-active clients removal
Clients with very low overall energy consumption are removed from the dataset. 
A minimum annual consumption threshold has been defined as follows:

$$P_{min} = 10 \, W \times 1 \, h \times N_{hours/year}$$

Where $N_{hours/year}$ represents the total number of working hours in the dataset period.

In [4]:
# Add year and date columns
df_workdays['Year'] = df_workdays['DateTime'].dt.year
df_workdays['Date'] = df_workdays['DateTime'].dt.date

# Count unique working days per year and calculate working half hours
working_days_per_year = df_workdays.groupby('Year')['Date'].nunique()
working_hhours_per_year = (working_days_per_year * 48).reset_index()
working_hhours_per_year.columns = ['Year', 'Working_Half_Hours']
working_hhours_per_year['MIN_THRESHOLD_KWH'] = 0.005 * working_hhours_per_year['Working_Half_Hours']
print(working_hhours_per_year)

# Convert KWh to numeric
df_workdays['KWh'] = pd.to_numeric(df_workdays['KWh'], errors='coerce')

# Calculate annual consumption per client per year
annual_consumption = df_workdays.groupby(['LCLid', 'Year'])['KWh'].sum().reset_index()
annual_consumption.columns = ['LCLid', 'Year', 'Annual_KWh']

# Merge threshold per year
annual_consumption = annual_consumption.merge(working_hhours_per_year[['Year', 'MIN_THRESHOLD_KWH']], on='Year')

# Find clients below threshold in ANY year
clients_below = annual_consumption[annual_consumption['Annual_KWh'] < annual_consumption['MIN_THRESHOLD_KWH']]['LCLid'].unique()

print(f"Clients removed (below threshold in at least one year): {len(clients_below)}")

# Remove those clients
df_clean = df_workdays[~df_workdays['LCLid'].isin(clients_below)].copy()
print(f"Remaining clients: {df_clean['LCLid'].nunique()}")
print(f"Remaining rows: {len(df_clean)}")
del df_workdays  # free up memory

   Year  Working_Half_Hours  MIN_THRESHOLD_KWH
0  2013               12144              60.72
Clients removed (below threshold in at least one year): 9
Remaining clients: 4525
Remaining rows: 54881969


## Outliers detection

In the following section, outliers within the dataset are removed. This has been done in order to clean the data before the clustering process, allowing the algorithm to find and identify more subtle changes in the daily load pattern. In order to do so, a Z-score method is implemented: it consists of calculating the mean value across all clients during a specific time slot of a certain day, within a limited number of days. Specifically the days selected for the comparison are those belonging to the rolling window time period: 9 days before and after the day under analysis. This has been done to avoid comparisons between different seasons or time periods across the whole year.

In [5]:
Z_THRESHOLD = 3.5
HALF_WINDOW = 9

df_clean['TimeSlot'] = df_clean['DateTime'].dt.hour * 2 + df_clean['DateTime'].dt.minute // 30
df_clean['Date'] = df_clean['DateTime'].dt.date

# Prepare border days
border_start['KWh'] = pd.to_numeric(border_start['KWh'], errors='coerce')
border_end['KWh'] = pd.to_numeric(border_end['KWh'], errors='coerce')
border_start['TimeSlot'] = border_start['DateTime'].dt.hour * 2 + border_start['DateTime'].dt.minute // 30
border_end['TimeSlot'] = border_end['DateTime'].dt.hour * 2 + border_end['DateTime'].dt.minute // 30
border_start['Date'] = border_start['DateTime'].dt.date
border_end['Date'] = border_end['DateTime'].dt.date

# Combine with border days for rolling window
df_rw = pd.concat([border_start, df_clean, border_end], ignore_index=True)

outlier_counts = 0
total_points = 0

for slot in range(48):
    slot_data = df_rw[df_rw['TimeSlot'] == slot].groupby(
        ['Date', 'LCLid'])['KWh'].mean().unstack('LCLid')
    
    mu = slot_data.rolling(window=2*HALF_WINDOW+1, center=True, min_periods=1).mean()
    sigma = slot_data.rolling(window=2*HALF_WINDOW+1, center=True, min_periods=1).std()
    z = (slot_data - mu) / sigma
    
    # Count outliers only on 2013 dates (not border days)
    z_2013 = z[z.index.map(lambda d: d.year == 2013)]
    outliers = z_2013.abs() > Z_THRESHOLD
    outlier_counts += outliers.values.sum()
    total_points += outliers.size

print(f"Total outliers: {outlier_counts} ({100*outlier_counts/total_points:.4f}%)")

Total outliers: 370688 (0.6732%)


### Outliers replacement

Once the outliers are located using the Z-score method previously described, each outlier is removed from the dataset and replaced with the mean value of the same half hour of the day across the rolling window time period.

In [6]:
df_clean_out = df_clean.copy()
df_clean_out['KWh_clean'] = df_clean_out['KWh']

for slot in range(48):
    slot_data = df_rw[df_rw['TimeSlot'] == slot].groupby(
        ['Date', 'LCLid'])['KWh'].mean().unstack('LCLid')
    
    mu = slot_data.rolling(window=2*HALF_WINDOW+1, center=True, min_periods=1).mean()
    sigma = slot_data.rolling(window=2*HALF_WINDOW+1, center=True, min_periods=1).std()
    z = (slot_data - mu) / sigma
    
    z_2013 = z[z.index.map(lambda d: d.year == 2013)]
    mu_2013 = mu[mu.index.map(lambda d: d.year == 2013)]
    
    outlier_mask = z_2013.abs() > Z_THRESHOLD
    replacement = mu_2013.where(outlier_mask)
    
    replacement_long = replacement.stack().reset_index()
    replacement_long.columns = ['Date', 'LCLid', 'replacement']
    
    df_slot = df_clean_out[df_clean_out['TimeSlot'] == slot].copy()
    df_slot = df_slot.merge(replacement_long, on=['Date', 'LCLid'], how='left')
    df_slot['KWh_clean'] = df_slot['replacement'].fillna(df_slot['KWh_clean'])
    
    df_clean_out.loc[df_clean_out['TimeSlot'] == slot, 'KWh_clean'] = df_slot['KWh_clean'].values

print("Outlier replacement done")

# Remove clients with any NaN
clients_no_nan = df_clean_out.groupby('LCLid')['KWh_clean'].apply(
    lambda x: x.isna().sum() == 0
)
valid_clients_kmeans = clients_no_nan[clients_no_nan].index
df_clean_out = df_clean_out[df_clean_out['LCLid'].isin(valid_clients_kmeans)].copy()

print(f"Clients after NaN removal: {df_clean_out['LCLid'].nunique()}")
print(f"Remaining NaN: {df_clean_out['KWh_clean'].isna().sum()}")

Outlier replacement done
Clients after NaN removal: 4525
Remaining NaN: 0


In [7]:
slots_per_client_day = df_clean_out.groupby(['LCLid', 'Date'])['TimeSlot'].nunique()
print(slots_per_client_day.describe())
print(f"\nGiorni con meno di 48 slot: {(slots_per_client_day < 48).sum()}")# Find complete days (all 48 slots)
complete_days = slots_per_client_day[slots_per_client_day == 48].reset_index()[['LCLid', 'Date']]

# Keep only complete days
df_clean_out = df_clean_out.merge(complete_days, on=['LCLid', 'Date'], how='inner')

# Keep only clients that still have 253 days
days_per_client = df_clean_out.groupby('LCLid')['Date'].nunique()
valid_clients = days_per_client[days_per_client == 253].index
df_clean_out = df_clean_out[df_clean_out['LCLid'].isin(valid_clients)].copy()

print(f"Remaining clients: {df_clean_out['LCLid'].nunique()}")

# Rebuild matrix
matrix = df_clean_out.pivot_table(
    index='LCLid',
    columns=['Date', 'TimeSlot'],
    values='KWh_clean'
)

client_ids = matrix.index.tolist()
print(f"Matrix shape: {matrix.shape}")
print(f"NaN count: {matrix.isna().sum().sum()}")

count    1.144383e+06
mean     4.795769e+01
std      1.247143e+00
min      1.000000e+00
25%      4.800000e+01
50%      4.800000e+01
75%      4.800000e+01
max      4.800000e+01
Name: TimeSlot, dtype: float64

Giorni con meno di 48 slot: 7606
Remaining clients: 1394
Matrix shape: (1394, 12144)
NaN count: 0


In [8]:
# Find complete days (all 48 slots)
complete_days = slots_per_client_day[slots_per_client_day == 48].reset_index()[['LCLid', 'Date']]

# Keep only complete days
df_clean_out = df_clean_out.merge(complete_days, on=['LCLid', 'Date'], how='inner')

# Keep only clients that still have 253 days
days_per_client = df_clean_out.groupby('LCLid')['Date'].nunique()
valid_clients = days_per_client[days_per_client == 253].index
df_clean_out = df_clean_out[df_clean_out['LCLid'].isin(valid_clients)].copy()

print(f"Remaining clients: {df_clean_out['LCLid'].nunique()}")

# Rebuild matrix
matrix = df_clean_out.pivot_table(
    index='LCLid',
    columns=['Date', 'TimeSlot'],
    values='KWh_clean'
)

client_ids = matrix.index.tolist()
print(f"Matrix shape: {matrix.shape}")
print(f"NaN count: {matrix.isna().sum().sum()}")

Remaining clients: 1394
Matrix shape: (1394, 12144)
NaN count: 0


## K-means

The k-means clustering algorithm has been used in this analysis with different number of clusters: from a minimum of 3 clusters to a maximum of 10. Once the clusters are defined, the results are investigated using a violin plot of silhout values. The higher the silhouette values is, the better that specific client belongs to its cluster.

In [9]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples

# Build average daily profile matrix (n_clients x 48)
matrix_avg = df_clean_out.groupby(['LCLid', 'TimeSlot'])['KWh_clean'].mean().unstack('TimeSlot')

client_ids_avg = matrix_avg.index.tolist()
X_avg = matrix_avg.values

print(f"Matrix shape: {matrix_avg.shape}")
print(f"NaN count: {matrix_avg.isna().sum().sum()}")

# K-means with silhouette violin plot
np.random.seed(4)

num_clusters = range(3, 11)
silhouette_scores_avg = []
silhouette_samples_list_avg = []
kmeans_models_avg = []
labels_list_avg = []

for k in num_clusters:
    selected_rows = np.random.choice(X_avg.shape[0], k, replace=False)
    init_centroids = X_avg[selected_rows, :]
    
    km = KMeans(n_clusters=k, init=init_centroids, n_init=1, random_state=4)
    labels = km.fit_predict(X_avg)
    
    s_samples = silhouette_samples(X_avg, labels)
    
    silhouette_scores_avg.append(s_samples.mean())
    silhouette_samples_list_avg.append(s_samples)
    kmeans_models_avg.append(km)
    labels_list_avg.append(labels)
    
    print(f"k={k} | Mean Silhouette: {s_samples.mean():.4f}")

# Violin plot
fig, ax = plt.subplots(figsize=(10, 5))

parts = ax.violinplot(silhouette_samples_list_avg, positions=list(num_clusters),
                      showmeans=True, showmedians=False,
                      showextrema=False)

for pc in parts['bodies']:
    pc.set_alpha(0.7)
parts['cmeans'].set_color('red')
parts['cmeans'].set_linewidth(2)

ax.set_xlabel('Number of clusters')
ax.set_ylabel('Silhouette score')
ax.set_title('Silhouette score distribution vs number of clusters (average profile)')
ax.set_xticks(list(num_clusters))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Matrix shape: (1394, 48)
NaN count: 0


NameError: name 'KMeans' is not defined

### Centroids plotting

In [ ]:
# Best k from violin plot - choose based on silhouette
best_k = 3
best_idx = best_k - 3
best_labels_avg = labels_list_avg[best_idx]
best_model_avg = kmeans_models_avg[best_idx]

time_labels = [f"{h:02d}:{m:02d}" for h in range(24) for m in [0, 30]]

plt.figure(figsize=(12, 5))
for i in range(best_k):
    cluster_mask = best_labels_avg == i
    cluster_client_ids = [client_ids_avg[j] for j in range(len(client_ids_avg)) if cluster_mask[j]]
    
    cluster_profile = df_clean_out[
        df_clean_out['LCLid'].isin(cluster_client_ids)
    ].groupby('TimeSlot')['KWh_clean'].mean()
    
    plt.plot(range(48), cluster_profile.values, 
             label=f'Cluster {i+1} (n={sum(cluster_mask)})', linewidth=2)

plt.xticks(range(0, 48, 4), time_labels[::4], rotation=45)
plt.xlabel('Time of day')
plt.ylabel('Average consumption (kWh)')
plt.title(f'Average daily load profile per cluster (k={best_k})')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## ACORN Comparison

In this section possible correspondences between clusters and the ACORN classification of end-users are investigated.

In [ ]:
# Load house info with ACORN classification
house_info = pd.read_csv('/kaggle/input/datasets/jeanmidev/smart-meters-in-london/informations_households.csv')
print(house_info.head())
print(house_info.columns.tolist())

In [ ]:
# Merge cluster labels with ACORN info
cluster_df = pd.DataFrame({
    'LCLid': client_ids_avg,
    'Cluster': best_labels_avg + 1  # 1-indexed
})

cluster_acorn = cluster_df.merge(
    house_info[['LCLid', 'Acorn_grouped']], 
    on='LCLid', 
    how='left'
)

print(cluster_acorn['Acorn_grouped'].value_counts())
print(f"\nNaN ACORN: {cluster_acorn['Acorn_grouped'].isna().sum()}")

# Cross-tabulation cluster vs ACORN
crosstab = pd.crosstab(
    cluster_acorn['Cluster'], 
    cluster_acorn['Acorn_grouped'],
    normalize='index'  # percentage per row
) * 100

print(crosstab.round(1))

In [ ]:
# Stacked bar chart - ACORN distribution per cluster
fig, ax = plt.subplots(figsize=(8, 5))

crosstab_plot = pd.crosstab(
    cluster_acorn['Cluster'],
    cluster_acorn['Acorn_grouped'],
    normalize='index'
) * 100

# Remove ACORN-U for clarity
crosstab_plot = crosstab_plot.drop(columns=['ACORN-U'], errors='ignore')

crosstab_plot.plot(
    kind='bar',
    stacked=True,
    ax=ax,
    color=['#d62728', '#2ca02c', '#1f77b4'],
    edgecolor='white',
    width=0.6
)

ax.set_xlabel('Cluster')
ax.set_ylabel('Percentage (%)')
ax.set_title('ACORN distribution per cluster (k=3)')
ax.set_xticklabels([f'Cluster {i}' for i in range(1, best_k+1)], rotation=0)
ax.legend(title='ACORN group', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Conclusions

The ACORN (A Classification Of Residential Neighbourhoods) classification seems to correspond to effective differences in daily load patterns. This is the most evident when looking at Cluster # 2, that exhibits the highest values of load usage amongst the others. It is important to note, however, that this cluster only contains 59 clients and for this reason it cannot successfully represent the typical distribution amongst London houses. 